# Seattle Permit Predictor — Cleaned Master Dataset
**Purpose:** Load raw CSVs, apply all agreed exclusion criteria, engineer features, merge review and comment aggregates, and write `master_dataset.csv` as the single shared starting point for all project branches.

**Grain:** One row per permit.

**Source data:** `C:\\Users\\flori\\Desktop\\data`

**Key conceptual decisions baked in:**
- Permits with no review records were approved on first submission
- Permits with review records were rejected at least once and required resubmission
- `totaldaysplanreview` <= 2 days = approved non-housing permits, excluded
- `totaldaysplanreview` > 1095 days = economically abandoned, excluded
- Comments represent project complexity, not approval/rejection signal
- `dwellingunittype` is the primary building type subcategory

In [18]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

SEED = 42
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = r'C:\Users\flori\Desktop\data'
OUTPUT_DIR  = r'C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output'
PERMITS_PATH  = os.path.join(DATA_DIR, 'building_permits.csv')
REVIEW_PATH   = os.path.join(DATA_DIR, 'plan_review.csv')
COMMENTS_PATH = os.path.join(DATA_DIR, 'plan_comments.csv')
OUTPUT_PATH   = os.path.join(OUTPUT_DIR, 'master_dataset.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration ready.')
print(f'  Source : {DATA_DIR}')
print(f'  Output : {OUTPUT_PATH}')

Configuration ready.
  Source : C:\Users\flori\Desktop\data
  Output : C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output\master_dataset.csv


In [19]:
# ── Cell 2: Load Raw CSVs ─────────────────────────────────────────────────────
df_permits  = pd.read_csv(PERMITS_PATH)
df_review   = pd.read_csv(REVIEW_PATH)
df_comments = pd.read_csv(COMMENTS_PATH)

print(f'Raw shapes:')
print(f'  df_permits  : {df_permits.shape}')
print(f'  df_review   : {df_review.shape}')
print(f'  df_comments : {df_comments.shape}')

Raw shapes:
  df_permits  : (189295, 40)
  df_review   : (253702, 46)
  df_comments : (26299, 8)


In [20]:
# ── Cell 3: Post-2020 Filter + Date Casting ───────────────────────────────────
# Cast date columns first, then apply cutoff

CUTOFF = pd.Timestamp('2020-01-01')

# Permits
permits_dates = [
    'applieddate', 'issueddate', 'expiresdate', 'completeddate',
    'readytoissuedate', 'initialreviewcompletedate', 'planreviewcompletedate'
]
for col in permits_dates:
    if col in df_permits.columns:
        df_permits[col] = pd.to_datetime(df_permits[col], errors='coerce')

# Review
review_dates = [
    'applieddate', 'reviewteamassigndate', 'reviewerassigndate',
    'reviewerfinishdate', 'initialreviewcompletedate', 'planreviewcompletedate',
    'readyissuedate', 'issueddate'
]
for col in review_dates:
    if col in df_review.columns:
        df_review[col] = pd.to_datetime(df_review[col], errors='coerce')

# Comments
df_comments['documentdate'] = pd.to_datetime(df_comments['documentdate'], errors='coerce')

# Apply cutoff
df_permits  = df_permits[df_permits['applieddate']   >= CUTOFF].copy()
df_review   = df_review[df_review['applieddate']     >= CUTOFF].copy()
df_comments = df_comments[df_comments['documentdate'] >= CUTOFF].copy()

print(f'Post-2020 shapes:')
print(f'  df_permits  : {df_permits.shape}')
print(f'  df_review   : {df_review.shape}')
print(f'  df_comments : {df_comments.shape}')

Post-2020 shapes:
  df_permits  : (40183, 40)
  df_review   : (196041, 46)
  df_comments : (26299, 8)


In [21]:
# ── Cell 4: Numeric Coercion on df_permits ────────────────────────────────────
numeric_cols = [
    'housingunits', 'housingunitsadded', 'housingunitsremoved',
    'estprojectcost', 'totaldaysplanreview', 'daysinitialplanreview',
    'daysplanreviewcity', 'daysoutcorrections', 'numberreviewcycles',
    'daysissuepermitcity', 'latitude', 'longitude'
]
for col in numeric_cols:
    if col in df_permits.columns:
        df_permits[col] = pd.to_numeric(df_permits[col], errors='coerce')

# Review numerics
for col in ['totaldaysplanreview', 'daysinitialplanreview', 'daysplanreviewcity',
            'housingunits', 'latitude', 'longitude', 'reviewcycle',
            'daysoutcorrections', 'numberreviewcycles']:
    if col in df_review.columns:
        df_review[col] = pd.to_numeric(df_review[col], errors='coerce')

df_comments['reviewcycle'] = pd.to_numeric(df_comments['reviewcycle'], errors='coerce')

print('Numeric coercion complete.')

Numeric coercion complete.


In [22]:
# ── Cell 5: Apply Exclusion Criteria to df_permits ────────────────────────────
# Document each exclusion step explicitly for auditability.

start_n = len(df_permits)
exclusion_log = []

def exclude(df, mask, reason):
    n_before = len(df)
    df = df[~mask].copy()
    n_excluded = n_before - len(df)
    exclusion_log.append((reason, n_excluded, len(df)))
    return df

# 1. Post-2020 already applied — drop 2026 incomplete permits
df_permits['app_year'] = df_permits['applieddate'].dt.year
df_permits = exclude(df_permits, df_permits['app_year'] == 2026,
                     '2026 permits (incomplete review cycles)')

# 2. Missing totaldaysplanreview
df_permits = exclude(df_permits, df_permits['totaldaysplanreview'].isna(),
                     'Missing totaldaysplanreview')

# 3. Negative totaldaysplanreview (data entry errors)
df_permits = exclude(df_permits, df_permits['totaldaysplanreview'] < 0,
                     'Negative totaldaysplanreview')

# 4. Zero totaldaysplanreview
df_permits = exclude(df_permits, df_permits['totaldaysplanreview'] == 0,
                     'Zero totaldaysplanreview')

# 5. <= 2 days: approved non-housing permits, not useful for modeling
df_permits = exclude(df_permits,
                     (df_permits['totaldaysplanreview'] > 0) &
                     (df_permits['totaldaysplanreview'] <= 2),
                     'totaldaysplanreview <= 2 days (approved non-housing, not useful)')

# 6. > 1095 days: economically abandoned projects
df_permits = exclude(df_permits, df_permits['totaldaysplanreview'] > 1095,
                     'totaldaysplanreview > 1095 days (economically abandoned)')

# 7. Missing or Unknown housingcategory
df_permits = exclude(df_permits,
                     df_permits['housingcategory'].isna() |
                     (df_permits['housingcategory'].str.strip().str.upper() == 'UNKNOWN'),
                     'Missing or Unknown housingcategory')

# 8. Missing or Unknown dwellingunittype
df_permits = exclude(df_permits,
                     df_permits['dwellingunittype'].isna() |
                     (df_permits['dwellingunittype'].str.strip().str.upper() == 'UNKNOWN'),
                     'Missing or Unknown dwellingunittype')

# 9. Fix negative housingunits → NaN (not an exclusion, a correction)
neg_units = (df_permits['housingunits'] < 0).sum()
df_permits.loc[df_permits['housingunits'] < 0, 'housingunits'] = np.nan

# 10. Fix zero estprojectcost → NaN
zero_cost = (df_permits['estprojectcost'] == 0).sum()
df_permits.loc[df_permits['estprojectcost'] == 0, 'estprojectcost'] = np.nan

# 11. Fix negative daysissuepermitcity → NaN
neg_issue = (df_permits['daysissuepermitcity'] < 0).sum()
df_permits.loc[df_permits['daysissuepermitcity'] < 0, 'daysissuepermitcity'] = np.nan

# ── Exclusion summary ─────────────────────────────────────────────────────────
print(f'── Exclusion log ────────────────────────────────────────────────────────')
print(f'  Starting population : {start_n:,}')
print(f'  {"Reason":<50} {"Excluded":>10} {"Remaining":>10}')
print(f'  {"─"*72}')
for reason, n_excl, n_rem in exclusion_log:
    print(f'  {reason:<50} {n_excl:>10,} {n_rem:>10,}')
print(f'\n  Value corrections (not exclusions):')
print(f'    housingunits negative → NaN    : {neg_units:,}')
print(f'    estprojectcost zero → NaN      : {zero_cost:,}')
print(f'    daysissuepermitcity neg → NaN  : {neg_issue:,}')
print(f'\n  Final population : {len(df_permits):,}')

── Exclusion log ────────────────────────────────────────────────────────
  Starting population : 40,183
  Reason                                               Excluded  Remaining
  ────────────────────────────────────────────────────────────────────────
  2026 permits (incomplete review cycles)                 1,876     38,307
  Missing totaldaysplanreview                            15,876     22,431
  Negative totaldaysplanreview                               80     22,351
  Zero totaldaysplanreview                                  123     22,228
  totaldaysplanreview <= 2 days (approved non-housing, not useful)        131     22,097
  totaldaysplanreview > 1095 days (economically abandoned)        179     21,918
  Missing or Unknown housingcategory                      6,771     15,147
  Missing or Unknown dwellingunittype                       946     14,201

  Value corrections (not exclusions):
    housingunits negative → NaN    : 41
    estprojectcost zero → NaN      : 45
    da

In [23]:
# ── Cell 6: Review Aggregates (one row per permit) ────────────────────────────
# Collapse df_review into per-permit features.
# Key reframe: presence of review records = project was rejected at least once.

# Filter review to permits that exist in our cleaned population
valid_permnums = set(df_permits['permitnum'].unique())
df_review_filtered = df_review[df_review['permitnum'].isin(valid_permnums)].copy()

print(f'Review rows for cleaned population: {len(df_review_filtered):,}')
print(f'Unique permits with review records : {df_review_filtered["permitnum"].nunique():,}')

# ── Aggregate ─────────────────────────────────────────────────────────────────
review_agg = (
    df_review_filtered
    .groupby('permitnum')
    .agg(
        # Rejection signal
        has_rejection            = ('permitnum', 'count'),           # placeholder, set to True below
        total_review_cycles      = ('reviewcycle', 'max'),
        n_correction_cycles      = ('reviewresultdesc',
                                    lambda x: (x == 'Corrections Required').sum()),
        any_corrections_required = ('reviewresultdesc',
                                    lambda x: (x == 'Corrections Required').any()),
        # Reviewer detail
        n_reviewers_assigned     = ('reviewer', 'nunique'),
        n_review_types           = ('reviewtype', 'nunique'),
        review_types_list        = ('reviewtype',
                                    lambda x: '|'.join(sorted(x.dropna().unique()))),
        # Complexity
        review_complexity_max    = ('reviewcomplexity',
                                    lambda x: x.dropna().iloc[-1]
                                    if x.notna().any() else np.nan),
    )
    .reset_index()
)

# has_rejection is True for all rows in this aggregation by definition
review_agg['has_rejection'] = True
review_agg['any_corrections_required'] = review_agg['any_corrections_required'].astype(bool)

print(f'\nReview aggregates shape: {review_agg.shape}')
print('\nSample:')
print(review_agg.head(3).to_string())

Review rows for cleaned population: 164,254
Unique permits with review records : 12,635

Review aggregates shape: (12635, 9)

Sample:
    permitnum  has_rejection  total_review_cycles  n_correction_cycles  any_corrections_required  n_reviewers_assigned  n_review_types                                                                                  review_types_list review_complexity_max
0  6421136-CN           True                    3                    5                      True                     8               6                      Addressing|ECA GeoTech|Energy|Ordinance/Structural|Side Sewer Conflict|Zoning                Full +
1  6508850-CN           True                    4                   15                      True                    11               9         Addressing|Drainage|ECA GeoTech|ECA Riparian|Energy|Fire|Ordinance/Structural|Parks|Zoning                Full +
2  6544617-CN           True                    6                   18                      True  

In [24]:
# ── Cell 7: Comment Aggregates (one row per permit) ───────────────────────────
# Collapse df_comments into per-permit features.
# Comments = complexity signal, not approval/rejection signal.

df_comments_filtered = df_comments[df_comments['permitnum'].isin(valid_permnums)].copy()

print(f'Comment rows for cleaned population : {len(df_comments_filtered):,}')
print(f'Unique permits with comment records : {df_comments_filtered["permitnum"].nunique():,}')

comment_agg = (
    df_comments_filtered
    .groupby('permitnum')
    .agg(
        has_comments              = ('permitnum', 'count'),   # placeholder, set to True below
        n_comments                = ('comment', 'count'),
        n_distinct_comment_subjects = ('commentsubject', 'nunique'),
        comment_reviewcycle_max   = ('reviewcycle', 'max'),
    )
    .reset_index()
)

comment_agg['has_comments'] = True

print(f'\nComment aggregates shape: {comment_agg.shape}')
print('\nSample:')
print(comment_agg.head(3).to_string())

Comment rows for cleaned population : 23,657
Unique permits with comment records : 665

Comment aggregates shape: (665, 5)

Sample:
    permitnum  has_comments  n_comments  n_distinct_comment_subjects  comment_reviewcycle_max
0  6698571-CN          True          48                           28                     3.00
1  6768034-CN          True          93                           29                     4.00
2  6802583-CN          True          81                           39                     4.00


In [25]:
# ── Cell 8: Feature Engineering on df_permits ─────────────────────────────────

# ── Date-derived features ─────────────────────────────────────────────────────
df_permits['app_year']    = df_permits['applieddate'].dt.year
df_permits['app_month']   = df_permits['applieddate'].dt.month
df_permits['app_quarter'] = df_permits['applieddate'].dt.quarter

# ── Zoning family ─────────────────────────────────────────────────────────────
def extract_zone_family(z):
    if pd.isna(z):
        return 'Unknown'
    z = str(z).strip().upper()
    for prefix in ['NR', 'LR', 'MR', 'HR', 'SF', 'NC', 'C1', 'C2', 'SM',
                   'IDM', 'IB', 'IG', 'IC', 'DH', 'DMC', 'DOC', 'MIO',
                   'PMM', 'SCM', 'PSM', 'ISRD']:
        if z.startswith(prefix):
            return prefix
    return 'Other'

df_permits['zone_family'] = df_permits['zoning'].apply(extract_zone_family)

# ── Log transforms ────────────────────────────────────────────────────────────
df_permits['log_estprojectcost']    = np.log1p(df_permits['estprojectcost'])
df_permits['log_housingunitsadded'] = np.log1p(df_permits['housingunitsadded'])
df_permits['log_target']            = np.log1p(df_permits['totaldaysplanreview'])

# ── Cap daysoutcorrections at 99th percentile ─────────────────────────────────
p99_doc = df_permits['daysoutcorrections'].quantile(0.99)
df_permits.loc[df_permits['daysoutcorrections'] > p99_doc, 'daysoutcorrections'] = p99_doc

print('Feature engineering complete.')
print(f'  zone_family value counts:')
print(df_permits['zone_family'].value_counts().to_string())
print(f'\n  log_target skewness : {df_permits["log_target"].skew():.3f}')
print(f'  raw target skewness : {df_permits["totaldaysplanreview"].skew():.3f}')

Feature engineering complete.
  zone_family value counts:
zone_family
NR         5477
SF         3293
LR         1691
Other      1023
NC          969
SM          316
DOC         259
IG          227
C1          210
DMC         207
IC          106
PSM          97
MIO          77
C2           68
DH           41
Unknown      32
MR           31
IDM          28
HR           18
PMM          17
IB           14

  log_target skewness : -0.109
  raw target skewness : 2.387


In [26]:
# ── Cell 9: Merge Review + Comment Aggregates onto df_permits ─────────────────

master = df_permits.copy()

# Merge review aggregates
master = master.merge(review_agg, on='permitnum', how='left')

# Merge comment aggregates
master = master.merge(comment_agg, on='permitnum', how='left')

# Fill False/0 for permits with no review or comment records
master['has_rejection']            = master['has_rejection'].fillna(False).astype(bool)
master['any_corrections_required'] = master['any_corrections_required'].fillna(False).astype(bool)
master['has_comments']             = master['has_comments'].fillna(False).astype(bool)
master['total_review_cycles']      = master['total_review_cycles'].fillna(0).astype(int)
master['n_correction_cycles']      = master['n_correction_cycles'].fillna(0).astype(int)
master['n_reviewers_assigned']     = master['n_reviewers_assigned'].fillna(0).astype(int)
master['n_review_types']           = master['n_review_types'].fillna(0).astype(int)
master['review_types_list']        = master['review_types_list'].fillna('')
master['n_comments']               = master['n_comments'].fillna(0).astype(int)
master['n_distinct_comment_subjects'] = master['n_distinct_comment_subjects'].fillna(0).astype(int)
master['comment_reviewcycle_max']  = master['comment_reviewcycle_max'].fillna(0)

print(f'Master dataset shape after merge: {master.shape}')
print(f'\nReview coverage:')
print(f'  has_rejection = True  : {master["has_rejection"].sum():,}  ({master["has_rejection"].mean()*100:.1f}%)')
print(f'  has_rejection = False : {(~master["has_rejection"]).sum():,}  ({(~master["has_rejection"]).mean()*100:.1f}%)')
print(f'\nComment coverage:')
print(f'  has_comments = True   : {master["has_comments"].sum():,}  ({master["has_comments"].mean()*100:.1f}%)')

Master dataset shape after merge: (14201, 59)

Review coverage:
  has_rejection = True  : 12,635  (89.0%)
  has_rejection = False : 1,566  (11.0%)

Comment coverage:
  has_comments = True   : 665  (4.7%)


In [27]:
# ── Cell 10: Column Selection + Ordering ──────────────────────────────────────
# Broad dataset — retain all useful columns for downstream branches.
# Reference columns kept but clearly grouped.

master_cols = [
    # ── Identity ──────────────────────────────────────────────────────────────
    'permitnum',
    'originaladdress1',
    'latitude',
    'longitude',

    # ── Target variable ───────────────────────────────────────────────────────
    'totaldaysplanreview',
    'log_target',

    # ── Primary model features ────────────────────────────────────────────────
    'permittypedesc',
    'dwellingunittype',
    'housingcategory',
    'zone_family',
    'zoning',
    'estprojectcost',
    'log_estprojectcost',
    'housingunitsadded',
    'log_housingunitsadded',
    'dependentbuilding',
    'app_year',
    'app_month',
    'app_quarter',

    # ── Review-derived features ───────────────────────────────────────────────
    'has_rejection',
    'total_review_cycles',
    'n_correction_cycles',
    'any_corrections_required',
    'n_reviewers_assigned',
    'n_review_types',
    'review_types_list',
    'review_complexity_max',

    # ── Comment-derived features ──────────────────────────────────────────────
    'has_comments',
    'n_comments',
    'n_distinct_comment_subjects',
    'comment_reviewcycle_max',

    # ── Post-hoc analysis columns (not for prediction model) ─────────────────
    'daysplanreviewcity',
    'daysinitialplanreview',
    'daysoutcorrections',
    'numberreviewcycles',
    'daysissuepermitcity',

    # ── Reference columns ─────────────────────────────────────────────────────
    'permitclass',
    'permitclassmapped',
    'permittypemapped',
    'statuscurrent',
    'housingunits',
    'housingunitsremoved',
    'applieddate',
    'issueddate',
    'completeddate',
    'contractorcompanyname',
]

# Keep only columns that exist in master
master_cols = [c for c in master_cols if c in master.columns]
master = master[master_cols].copy()

print(f'Final master dataset: {master.shape[0]:,} rows × {master.shape[1]} columns')
print(f'\nColumns included ({len(master_cols)}):')
for c in master_cols:
    print(f'  {c}')

Final master dataset: 14,201 rows × 46 columns

Columns included (46):
  permitnum
  originaladdress1
  latitude
  longitude
  totaldaysplanreview
  log_target
  permittypedesc
  dwellingunittype
  housingcategory
  zone_family
  zoning
  estprojectcost
  log_estprojectcost
  housingunitsadded
  log_housingunitsadded
  dependentbuilding
  app_year
  app_month
  app_quarter
  has_rejection
  total_review_cycles
  n_correction_cycles
  any_corrections_required
  n_reviewers_assigned
  n_review_types
  review_types_list
  review_complexity_max
  has_comments
  n_comments
  n_distinct_comment_subjects
  comment_reviewcycle_max
  daysplanreviewcity
  daysinitialplanreview
  daysoutcorrections
  numberreviewcycles
  daysissuepermitcity
  permitclass
  permitclassmapped
  permittypemapped
  statuscurrent
  housingunits
  housingunitsremoved
  applieddate
  issueddate
  completeddate
  contractorcompanyname


In [28]:
# ── Cell 11: Final Null Audit ─────────────────────────────────────────────────

print(f'── Null audit on master dataset ({master.shape[0]:,} rows) ──────────────────────')
print(f'  {"Column":<35} {"Null%":>7}  {"Null n":>8}')
print(f'  {"─"*55}')
for col in master.columns:
    n_null  = master[col].isna().sum()
    pct     = master[col].isna().mean() * 100
    flag    = '  ◀ review' if pct > 10 else ''
    print(f'  {col:<35} {pct:>6.1f}%  {n_null:>8,}{flag}')

print(f'\n── Target variable summary ──────────────────────────────────────────────')
print(f'  min    : {master["totaldaysplanreview"].min():.0f} days')
print(f'  median : {master["totaldaysplanreview"].median():.0f} days')
print(f'  mean   : {master["totaldaysplanreview"].mean():.1f} days')
print(f'  max    : {master["totaldaysplanreview"].max():.0f} days')
print(f'  skew   : {master["totaldaysplanreview"].skew():.3f} (raw)')
print(f'  skew   : {master["log_target"].skew():.3f} (log)')

print(f'\n── Key feature distributions ────────────────────────────────────────────')
for col in ['permittypedesc', 'dwellingunittype', 'housingcategory', 'zone_family']:
    print(f'\n  {col}')
    print(master[col].value_counts().to_string())

── Null audit on master dataset (14,201 rows) ──────────────────────
  Column                                Null%    Null n
  ───────────────────────────────────────────────────────
  permitnum                              0.0%         0
  originaladdress1                       0.1%        15
  latitude                               0.1%        19
  longitude                              0.1%        19
  totaldaysplanreview                    0.0%         0
  log_target                             0.0%         0
  permittypedesc                         0.0%         7
  dwellingunittype                       0.0%         0
  housingcategory                        0.0%         0
  zone_family                            0.0%         0
  zoning                                 0.2%        32
  estprojectcost                         0.3%        45
  log_estprojectcost                     0.3%        45
  housingunitsadded                      1.7%       237
  log_housingunitsadded          

In [29]:
# ── Cell 11.5: Final Cleanup Before Write ─────────────────────────────────────
# Drop contractorcompanyname — 95%+ null, no modeling value
master = master.drop(columns=['contractorcompanyname'])
print('── Dropped contractorcompanyname (95.3% null, no modeling value)')

# ── Document multi-value dwellingunittype entries ─────────────────────────────
# Some entries are pipe/comma-separated combinations e.g.
# "Accessory Dwelling Attached, Detached Single-Family"
# These are valid city classifications, not data errors.
# Downstream branches doing categorical encoding should be aware.

multi_val = master[master['dwellingunittype'].str.contains(',', na=False)]
print(f'\n── dwellingunittype multi-value entries: {len(multi_val):,} rows')
print('   These are combined unit type classifications from the city.')
print('   Encoding strategy should be decided per branch (dummies vs. treat as-is).')
print()
print(multi_val['dwellingunittype'].value_counts().to_string())

print(f'\n── Master dataset ready to write: {master.shape[0]:,} rows × {master.shape[1]} columns')

── Dropped contractorcompanyname (95.3% null, no modeling value)

── dwellingunittype multi-value entries: 1,361 rows
   These are combined unit type classifications from the city.
   Encoding strategy should be decided per branch (dummies vs. treat as-is).

dwellingunittype
Accessory Dwelling Attached, Detached Single-Family                                 922
Accessory Dwelling Attached, Accessory Dwelling Detached, Detached Single-Family    157
Accessory Dwelling Detached, Detached Single-Family                                 156
Accessory Dwelling Attached, Accessory Dwelling Detached                             42
Detached Single-Family, Accessory Dwelling Attached                                  30
Accessory Dwelling Attached, Townhouse                                               24
Accessory Dwelling Attached, Rowhouse                                                12
Accessory Live/Work, Townhouse                                                        5
Accessory Dwelling D

In [30]:
# ── Cell 12: Write master_dataset.csv ────────────────────────────────────────

master.to_csv(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)

print(f'── master_dataset.csv written successfully ───────────────────────────────')
print(f'  Path    : {OUTPUT_PATH}')
print(f'  Rows    : {len(master):,}')
print(f'  Columns : {master.shape[1]}')
print(f'  Size    : {size_mb:.2f} MB')
print(f'\n── Column reference for downstream branches ─────────────────────────────')
print(f'  TARGET     : totaldaysplanreview, log_target')
print(f'  MODEL FEATS: permittypedesc, dwellingunittype, housingcategory,')
print(f'               zone_family, zoning, log_estprojectcost,')
print(f'               log_housingunitsadded, dependentbuilding,')
print(f'               app_year, app_month, app_quarter, latitude, longitude')
print(f'  REVIEW FEATS: has_rejection, total_review_cycles, n_correction_cycles,')
print(f'               any_corrections_required, n_reviewers_assigned,')
print(f'               n_review_types, review_types_list, review_complexity_max')
print(f'  COMMENT FEATS: has_comments, n_comments, n_distinct_comment_subjects,')
print(f'               comment_reviewcycle_max')
print(f'  POST-HOC   : daysplanreviewcity, daysinitialplanreview,')
print(f'               daysoutcorrections, numberreviewcycles, daysissuepermitcity')
print(f'  REFERENCE  : permitclass, permitclassmapped, permittypemapped,')
print(f'               statuscurrent, housingunits, housingunitsremoved,')
print(f'               applieddate, issueddate, completeddate, contractorcompanyname')

── master_dataset.csv written successfully ───────────────────────────────
  Path    : C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output\master_dataset.csv
  Rows    : 14,201
  Columns : 45
  Size    : 5.59 MB

── Column reference for downstream branches ─────────────────────────────
  TARGET     : totaldaysplanreview, log_target
  MODEL FEATS: permittypedesc, dwellingunittype, housingcategory,
               zone_family, zoning, log_estprojectcost,
               log_housingunitsadded, dependentbuilding,
               app_year, app_month, app_quarter, latitude, longitude
  REVIEW FEATS: has_rejection, total_review_cycles, n_correction_cycles,
               any_corrections_required, n_reviewers_assigned,
               n_review_types, review_types_list, review_complexity_max
  COMMENT FEATS: has_comments, n_comments, n_distinct_comment_subjects,
               comment_reviewcycle_max
  POST-HOC   : daysplanreviewcity, daysinitialplanreview,
              